<a href="https://colab.research.google.com/github/CristianCarrereAlvarez/monitor-mercado-laboral/blob/main/SMLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitor Mercado Laboral Chile

Código en este repo; datos en Google Drive. Ver `CLAUDE.md` para el contexto completo.

**Orden de uso:** correr la sección 1 entera y en orden (celdas 1.1 → 1.3) al abrir la
sesión. Después, la sección 2 para capturar y la 3 para consolidar. La sección 4 es
opcional.

---

## ⚠️ La regla que no se puede romper

`scraper_v9.py` escribe en `crudo/` **relativo al directorio actual**. Si lo corrés
parado en `/content/repo`, el crudo queda en el clon efímero: se pierde al reiniciar
la sesión, `.gitignore` lo oculta y nada te avisa. El síntoma es que consolidás y el
área que acabás de correr no aparece.

Por eso **toda** celda de captura y de consolidación empieza con `%cd $DATOS`:

```
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "<Área>"
```

`PYTHONPATH` permite importar `carreras_sies_2026.py` desde el repo mientras el
working dir está en Drive. `-u` desactiva el buffering para ver el progreso en vivo.

---

## 1. Preparación de la sesión

Correr las tres celdas en orden. Colab corta por inactividad (~90 min), así que
hay que repetirlas al reabrir.

### 1.1 Dependencias

In [ ]:
!pip install -q playwright nest_asyncio
!playwright install --with-deps chromium

### 1.2 Drive — define `DATOS`

Todas las celdas siguientes dependen de esta variable. Si salta un `NameError`,
es que esta celda no se corrió.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATOS = '/content/drive/MyDrive/monitor_mercado_laboral'
os.makedirs(f'{DATOS}/crudo', exist_ok=True)
os.makedirs(f'{DATOS}/maestras', exist_ok=True)

print('crudo   :', sorted(os.listdir(f'{DATOS}/crudo')))
print('maestras:', sorted(os.listdir(f'{DATOS}/maestras')))

### 1.3 Repositorio

Clona si no está, y trae la última versión siempre. El repo es la fuente de
verdad del código; los datos nunca viven acá.

In [ ]:
import os, subprocess

REPO = 'https://github.com/CristianCarrereAlvarez/monitor-mercado-laboral.git'

if not os.path.exists('/content/repo'):
    !git clone $REPO /content/repo
else:
    !git -C /content/repo pull --ff-only

print(subprocess.run(['git','-C','/content/repo','log','--oneline','-1'],
                     capture_output=True, text=True).stdout)
print(sorted(os.listdir('/content/repo')))

**Si `git pull` falla** con *"untracked working tree file would be overwritten"*,
re-cloná limpio con esta celda (es seguro: en el repo no hay datos):

In [ ]:
import shutil
shutil.rmtree('/content/repo', ignore_errors=True)
!git clone $REPO /content/repo
!ls /content/repo

---

## 2. Captura por área

> ⛔ **Desde agosto 2026 esto no funciona en Colab.** Akamai bloquea los
> rangos de datacenter: `trabajando.cl` devuelve 403 en todo, incluida la
> portada. La captura hay que correrla desde una máquina con IP
> residencial. Las secciones 3 y 4 sí andan acá — no tocan la red.

> Desde una terminal, con una sola línea:
> ```
> ~/monitor-mercado-laboral/capturar.sh "Agropecuaria"
> ```
> El envoltorio resuelve la carpeta de datos, verifica que exista y elige
> el modo de sesión. No hay que hacer `cd` a nada ni recordar el
> `PYTHONPATH`.

Una celda por área, **ordenadas de menor a mayor** cantidad de términos:
conviene empezar chico para detectar problemas barato.

Cada corrida es **reanudable**: si se corta, relanzar lo mismo. Lee el
JSONL existente y baja solo lo que falta. Si el sitio rechaza, aborta con
un mensaje explícito en vez de decir que terminó bien.

In [ ]:
# Derecho — 1 término
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Derecho"

In [ ]:
# Humanidades — 3 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Humanidades"

In [ ]:
# Agropecuaria — 9 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Agropecuaria"

In [ ]:
# Arte y Arquitectura — 12 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Arte y Arquitectura"

In [ ]:
# Ciencias Sociales — 14 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Ciencias Sociales"

In [ ]:
# Educación — 14 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Educación"

In [ ]:
# Ciencias Básicas — 15 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Ciencias Básicas"

In [ ]:
# Salud — 22 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Salud"

In [ ]:
# Administración y Comercio — 28 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Administración y Comercio"

In [ ]:
# Tecnología — 81 términos
%cd $DATOS
!PYTHONPATH=/content/repo python -u /content/repo/scraper_v9.py "Tecnología"

---

## 3. Consolidación

Lee todos los `crudo_*.jsonl` y hace upsert sobre las ocho maestras. Es
**idempotente**: se puede correr las veces que sea. Si cambia un criterio de
parseo, se reconsolida — nunca se re-scrapea por eso.

Las columnas manuales que agregues a mano (`carrera_sies`, `rubro`, `tamano`…)
**sobreviven** a cada corrida.

In [ ]:
%cd /content/repo
!git pull --ff-only
%cd $DATOS
!python /content/repo/consolidar.py --crudo crudo --maestras maestras

---

## 4. Verificaciones rápidas

Opcionales. Sobre las maestras ya consolidadas.

### 4.1 Distribución de carreras declaradas

Debería ser bimodal: valores chicos y después un salto a los avisos genéricos.
`UMBRAL_AVISO_GENERICO = 30` corta en el hueco.

In [ ]:
import pandas as pd
a = pd.read_csv(f'{DATOS}/maestras/avisos.csv')
print(a.n_carreras_declaradas.value_counts().sort_index().to_string())

### 4.2 Los avisos genéricos

Un puñado de empleadores declara casi todo el catálogo. Aparecen en la búsqueda
de cualquier carrera y hay que excluirlos de cualquier análisis por carrera.

In [ ]:
print(a[a.n_carreras_declaradas > 30]
      [['aviso_id','empresa_id','titulo','comuna','n_carreras_declaradas','n_instituciones']]
      .sort_values('n_carreras_declaradas', ascending=False)
      .to_string(index=False, max_colwidth=45))

### 4.3 Concentración por empleador

**Leer antes de cualquier agregado por área.** En Derecho un solo empleador
aportó el 35% de los avisos con su boilerplate institucional. El conteo de
avisos mide publicación, no demanda.

In [ ]:
top = a.empresa_id.value_counts()
n = len(a)
for k in (1, 3, 10):
    print(f'top {k:2d} empleadores: {top.head(k).sum()*100/n:.1f}% de {n} avisos')
print()
print(a.groupby('empresa_id').size().sort_values(ascending=False).head(10).to_string())

### 4.4 Ruta A — término de búsqueda → SIES

**Sobre-atribuye por diseño.** El término es lo que se buscó, no lo que el aviso
declara: en Derecho los 342 avisos quedan etiquetados «Derecho», incluidos los
que no son jurídicos. Es señal de contexto, no clasificación.

`n_terminos_sin_mapeo > 0` significa que el crudo tiene términos que ya no están
en el catálogo — detector de deriva.

In [ ]:
at = pd.read_csv(f'{DATOS}/maestras/aviso_termino.csv')
print(at.groupby(['termino_busqueda','carrera_sies','areas_sies']).size()
        .sort_values(ascending=False).head(20).to_string())
print()
print('avisos con términos sin mapeo:', int((a.n_terminos_sin_mapeo > 0).sum()))

### 4.5 Panel longitudinal — duración de vacante

**No mezclar las tres calidades.** Solo `observada` es una medición real;
`cota_superior` es un techo y `censurada` sigue viva. Hacen falta al menos dos
corridas del mismo aviso para tener una sola medición.

In [ ]:
print(a.calidad_duracion.value_counts().to_string())
print()
obs = a[a.calidad_duracion == 'observada']['dias_publicado_hasta_baja'].dropna()
print(obs.describe().to_string() if len(obs) else
      'sin duraciones observadas todavía — hace falta una segunda corrida')

### 4.6 Prioridad de homologación

Ordenado por `n_avisos_especificos`, que ignora los avisos genéricos. Es el orden
correcto para completar a mano `carrera_sies` en `carreras_trabajando.csv`.

In [ ]:
ct = pd.read_csv(f'{DATOS}/maestras/carreras_trabajando.csv')
print(ct.head(30)[['carrera_trabajando','n_avisos_especificos','n_avisos_acum']]
        .to_string(index=False))